In [ ]:
code = 'SRE_PS_W_Straddle'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SREW_SHIFT(bt, start_time, end_time, orderside, method, om, divider, movement1, movement2):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)

        entry_time = ''
        from_candle_close = True if method == 'CC' else False

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        if get_strike(ce_scrip) <= get_strike(pe_scrip):
            start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
            movement = movement2
            check_non_std_trades = False
        else:
            entry_time = start_dt
            movement = movement1
            check_non_std_trades = True

            premium = ce_price + pe_price
            ce_start_dt, pe_start_dt = start_dt, start_dt

        ## Check Non Straddle Trades ##
        trades = []
        shifting_pnl, eod_ce_pnl, eod_pe_pnl, ps_total_pnl = 0, 0, 0, 0
        exit_time = end_dt

        if not check_non_std_trades:
            for _ in range(max_re+1):
                trades.extend(['', '', '', '', '', '', '', '', '', '', 0])
            trades.extend([shifting_pnl, eod_ce_pnl, eod_pe_pnl, ps_total_pnl])
        else:
            for re in range(max_re+1):

                divider_price = cal_percent(premium, divider)
                move_price = cal_percent(premium, movement)

                start_dt = max(ce_start_dt, pe_start_dt)
                t_ce, t_pe = bt.get_straddle_data(start_dt, end_dt, ce_scrip, pe_scrip, seperate=True)
                ce_data, pe_data = t_ce.copy(), t_pe.copy()

                _, _, _, _, ce_sl_time, _ = bt.sl_check_by_given_data(ce_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)
                _, _, _, _, pe_sl_time, _ = bt.sl_check_by_given_data(pe_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)

                ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
                pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m

                if ce_sl_time < pe_sl_time:

                    ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                    pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']

                    pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                    shifting_pnl += pnl
                    trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, 'PE', ce_sl_time, pe_price_at_sl, ce_price_at_sl])

                    target_price = pe_price_at_sl + divider_price
                    pe_scrip, pe_price, _, pe_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='PE')
                    if pe_scrip is None:
                        exit_time = ce_sl_time
                        trades.append(pnl)
                        break
                    else:
                        if get_strike(ce_scrip) > get_strike(pe_scrip):
                            trades.append(pnl)
                            premium = premium - pe_price_at_sl + pe_price
                        else:
                            ce_pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                            shifting_pnl += ce_pnl
                            pnl += ce_pnl
                            trades.append(pnl)
                            start_dt = ce_sl_time
                            movement = movement2
                            check_non_std_trades = False
                            ce_scrip, pe_scrip = None, None
                            ce_price, pe_price = '', ''
                            break

                elif pe_sl_time < ce_sl_time:

                    ce_price_at_sl = bt.options_data.loc[(pe_sl_time, ce_scrip), 'close']
                    pe_price_at_sl = bt.options_data.loc[(pe_sl_time, pe_scrip), 'close']

                    pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                    shifting_pnl += pnl
                    trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, 'CE', pe_sl_time, ce_price_at_sl, pe_price_at_sl])

                    target_price = ce_price_at_sl + divider_price
                    ce_scrip, ce_price, _, ce_start_dt = bt.get_strike(pe_sl_time, end_dt, target=target_price, obove_target_only=True, only='CE')
                    if ce_scrip is None: 
                        exit_time = pe_sl_time
                        trades.append(pnl)
                        break
                    else:
                        if get_strike(ce_scrip) > get_strike(pe_scrip):
                            trades.append(pnl)
                            premium = premium - ce_price_at_sl + ce_price
                        else:
                            pe_pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                            shifting_pnl += pe_pnl
                            pnl += pe_pnl
                            trades.append(pnl)
                            start_dt = pe_sl_time
                            movement = movement2
                            check_non_std_trades = False
                            ce_scrip, pe_scrip = None, None
                            ce_price, pe_price = '', ''
                            break

                else:
                    if ce_sl_time != end_dt_1m:

                        ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                        pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']

                        ce_pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                        shifting_pnl += ce_pnl

                        pe_pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                        shifting_pnl += pe_pnl

                        trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, 'BOTH', ce_sl_time, ce_price_at_sl, pe_price_at_sl, ce_pnl+pe_pnl])

                        target_price = pe_price_at_sl + divider_price
                        pe_scrip, pe_price, _, pe_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='PE')
                        if pe_scrip is None:
                            ce_scrip, exit_time = None, None
                            break

                        target_price = ce_price_at_sl + divider_price
                        ce_scrip, ce_price, _, ce_start_dt = bt.get_strike(ce_sl_time, end_dt, target=target_price, obove_target_only=True, only='CE')
                        if ce_scrip is None:
                            pe_scrip, exit_time = None, None
                            break

                        if get_strike(ce_scrip) > get_strike(pe_scrip):
                            premium = premium - ce_price_at_sl + ce_price - pe_price_at_sl + pe_price
                        else:
                            start_dt = ce_sl_time
                            movement = movement2
                            check_non_std_trades = False
                            ce_scrip, pe_scrip = None, None
                            ce_price, pe_price = '', ''
                            break

                    else:
                        eod_ce_price = bt.options.loc[(bt.options['scrip'] == ce_scrip) & (bt.options['date_time'] <= ce_sl_time), 'close'].iloc[-1]
                        eod_ce_pnl = round(ce_price - eod_ce_price - bt.Cal_slipage(ce_price), 2)

                        eod_pe_price = bt.options.loc[(bt.options['scrip'] == pe_scrip) & (bt.options['date_time'] <= pe_sl_time), 'close'].iloc[-1]
                        eod_pe_pnl = round(pe_price - eod_pe_price - bt.Cal_slipage(pe_price), 2)

                        trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, divider_price, '', '', '', '', eod_ce_pnl + eod_pe_pnl])

                        ce_scrip, pe_scrip = None, None
                        ce_price, pe_price = '', ''
                        break

            if ce_scrip is None and pe_scrip is not None:
                eod_ce_pnl = 0
                eod_pe_price = bt.options.loc[(bt.options['scrip'] == pe_scrip) & (bt.options['date_time'] <= exit_time), 'close'].iloc[-1]
                eod_pe_pnl = round(pe_price - eod_pe_price - bt.Cal_slipage(pe_price), 2)
                trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, '', '', '', '', '', '', eod_ce_pnl + eod_pe_pnl])
            elif ce_scrip is not None and pe_scrip is None:
                eod_pe_pnl = 0
                eod_ce_price = bt.options.loc[(bt.options['scrip'] == ce_scrip) & (bt.options['date_time'] <= exit_time), 'close'].iloc[-1]
                eod_ce_pnl = round(ce_price - eod_ce_price - bt.Cal_slipage(ce_price), 2)
                trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, '', '', '', '', '', '', eod_ce_pnl + eod_pe_pnl])
            elif ce_scrip is None and pe_scrip is None:
                trades.extend([ce_scrip, ce_price, pe_scrip, pe_price, '', '', '', '', '', '', 0])

            ps_total_pnl = shifting_pnl + eod_ce_pnl + eod_pe_pnl
            for _ in range(max_re - re - 1):
                trades.extend(['', '', '', '', '', '', '', '', '', '', 0])
            trades.extend([shifting_pnl, eod_ce_pnl, eod_pe_pnl, ps_total_pnl])

        ## Check Straddle Trades ##
        trades1 = []
        shifting_pnl, eod_ce_pnl, eod_pe_pnl, total_pnl = 0, 0, 0, 0
        exit_time = end_dt

        if not check_non_std_trades:
            for re in range(std_re_entries+1):
                ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=0)
                if ce_scrip is None: return None

                entry_time = entry_time if entry_time else start_dt

                premium = ce_price + pe_price
                move_price = cal_percent(premium, movement)

                t_ce, t_pe = bt.get_straddle_data(start_dt, end_dt, ce_scrip, pe_scrip, seperate=True)
                ce_data, pe_data = t_ce.copy(), t_pe.copy()

                _, _, _, _, ce_sl_time, _ = bt.sl_check_by_given_data(ce_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)
                _, _, _, _, pe_sl_time, _ = bt.sl_check_by_given_data(pe_data, sl_price=move_price, orderside=orderside, from_candle_close=from_candle_close)

                ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
                pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m

                if ce_sl_time < pe_sl_time:
                    start_dt = ce_sl_time

                    ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                    pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']

                    ce_pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                    shifting_pnl += ce_pnl

                    pe_pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                    shifting_pnl += pe_pnl

                    trades1.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, '', 'PE', ce_sl_time, ce_price_at_sl, pe_price_at_sl, ce_pnl+pe_pnl])

                elif pe_sl_time < ce_sl_time:
                    start_dt = pe_sl_time

                    ce_price_at_sl = bt.options_data.loc[(pe_sl_time, ce_scrip), 'close']
                    pe_price_at_sl = bt.options_data.loc[(pe_sl_time, pe_scrip), 'close']

                    ce_pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                    shifting_pnl += ce_pnl

                    pe_pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                    shifting_pnl += pe_pnl

                    trades1.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, '', 'CE', pe_sl_time, ce_price_at_sl, pe_price_at_sl, ce_pnl+pe_pnl])

                else:
                    if ce_sl_time != end_dt_1m:
                        start_dt = ce_sl_time

                        ce_price_at_sl = bt.options_data.loc[(ce_sl_time, ce_scrip), 'close']
                        pe_price_at_sl = bt.options_data.loc[(ce_sl_time, pe_scrip), 'close']

                        ce_pnl = round((ce_price - ce_price_at_sl) - bt.Cal_slipage(ce_price), 2)
                        shifting_pnl += ce_pnl

                        pe_pnl = round((pe_price - pe_price_at_sl) - bt.Cal_slipage(pe_price), 2)
                        shifting_pnl += pe_pnl

                        trades1.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, '', 'BOTH', ce_sl_time, ce_price_at_sl, pe_price_at_sl, ce_pnl+pe_pnl])
                        
                    else:
                        eod_ce_price = bt.options.loc[(bt.options['scrip'] == ce_scrip) & (bt.options['date_time'] <= ce_sl_time), 'close'].iloc[-1]
                        eod_ce_pnl = round(ce_price - eod_ce_price - bt.Cal_slipage(ce_price), 2)

                        eod_pe_price = bt.options.loc[(bt.options['scrip'] == pe_scrip) & (bt.options['date_time'] <= pe_sl_time), 'close'].iloc[-1]
                        eod_pe_pnl = round(pe_price - eod_pe_price - bt.Cal_slipage(pe_price), 2)

                        trades1.extend([ce_scrip, ce_price, pe_scrip, pe_price, premium, '', '', '', '', '', eod_ce_pnl + eod_pe_pnl])
                        exit_time = end_dt
                        break

            total_pnl = shifting_pnl + eod_ce_pnl + eod_pe_pnl + ps_total_pnl
            for _ in range(std_re_entries - re):
                trades1.extend(['', '', '', '', '', '', '', '', '', '', 0])
            trades1.extend([shifting_pnl, eod_ce_pnl, eod_pe_pnl, total_pnl])

        else:
            total_pnl = shifting_pnl + eod_ce_pnl + eod_pe_pnl + ps_total_pnl
            for _ in range(std_re_entries+1):
                trades1.extend(['', '', '', '', '', '', '', '', '', '', 0])
            trades1.extend([shifting_pnl, eod_ce_pnl, eod_pe_pnl, total_pnl])

        return [code, bt.index, start_time, end_time, orderside, method, om, divider, movement1, movement2, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time.time(), future_price] + trades + trades1

    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, orderside, method, om, divider, movement1, movement2])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            max_re, re_entries, std_re_entries = 20, 20, 20

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_OM/P_Divider/P_Movement1/P_Movement2/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future'

            for r in range(max_re+1):
                log_cols += f'/CE{r}.Scrip/CE{r}.Price/PE{r}.Scrip/PE{r}.Price/Premium{r}/Divider{r}/ShiftLeg{r}/ShiftLeg{r}.Time/ShiftLeg{r}.Price.AT.SL/NonShift{r}.Price.At.SL/Shift{r}.PNL'
            log_cols += f'/Shifted.PNL/EOD.CE.PNL/EOD.PE.PNL/PS.Total.PNL'

            for r in range(std_re_entries+1):
                log_cols += f'/S.CE{r}.Scrip/S.CE{r}.Price/S.PE{r}.Scrip/S.PE{r}.Price/S.Premium{r}/S.Divider{r}/S.ShiftLeg{r}/S.ShiftLeg{r}.Time/S.ShiftLeg{r}.Price.AT.SL/S.NonShift{r}.Price.At.SL/S.Shift{r}.PNL'
            log_cols += f'/S.Shifted.PNL/S.EOD.CE.PNL/S.EOD.PE.PNL/Total.PNL'
            log_cols = log_cols.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SREW_SHIFT(wbt, row['entry_time'], row['exit_time'], row['orderside'], row['method'], row['om'], row['divider'], row['movement1'], row['movement2']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del wbt
                    gc.collect()

                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))